# Import thư viện, Khởi tạo và Metric

In [1]:
import pandas as pd
import numpy as np
import os

from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import (
    classification_report, accuracy_score, balanced_accuracy_score,
    f1_score, precision_score, recall_score, matthews_corrcoef,
    cohen_kappa_score, confusion_matrix
)

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

import warnings
warnings.filterwarnings('ignore')

device =  torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cuda


In [2]:
class TimeSeriesBiLSTMDataset(Dataset):
    def __init__(self, X, y):
        # X: (số mẫu, 4 phase, số feature mỗi phase)
        self.X = torch.FloatTensor(X)
        self.y = torch.LongTensor(y)
    
    def __len__(self):
        return len(self.X)
    
    def __getitem__(self, idx):
        # Trả về 1 chuỗi 4 phase + nhãn tương ứng
        return self.X[idx], self.y[idx]

class TimeSeriesBiLSTM(nn.Module):
    def __init__(
        self,
        input_dim_per_phase,
        hidden_dim=64,
        num_layers=2,
        num_classes=3,
        dropout=0.3,
        bidirectional=True
    ):
        super().__init__()

        self.lstm = nn.LSTM(
            input_size=input_dim_per_phase,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0,
            bidirectional=bidirectional
        )

        multiplier = 2 if bidirectional else 1

        self.fc = nn.Linear(hidden_dim * multiplier, num_classes)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        # x: (batch, seq_len=4, features)
        out, (hn, cn) = self.lstm(x)

        # Lấy output tại phase cuối
        out = out[:, -1, :]
        out = self.dropout(out)

        return self.fc(out)



In [3]:
# Metrics
def gmean_score(y_true, y_pred):
    cm = confusion_matrix(y_true, y_pred)
    per_class = []
    for i in range(cm.shape[0]):
        tp = cm[i,i]
        fn = cm[i].sum() - tp
        fp = cm[:,i].sum() - tp
        tn = cm.sum() - tp - fn - fp
        sens = tp / (tp + fn) if (tp + fn) > 0 else 0
        spec = tn / (tn + fp) if (tn + fp) > 0 else 0
        per_class.append(np.sqrt(sens * spec))
    return np.prod(per_class) ** (1/len(per_class)) if per_class else 0

def gmean_per_class(y_true, y_pred, target_class):
    cm = confusion_matrix(y_true, y_pred)
    i = target_class
    tp = cm[i,i]
    fn = cm[i].sum() - tp
    fp = cm[:,i].sum() - tp
    tn = cm.sum() - tp - fn - fp

    recall = tp / (tp + fn) if (tp + fn) > 0 else 0
    specificity = tn / (tn + fp) if (tn + fp) > 0 else 0
    return np.sqrt(recall * specificity)

def print_results(version_name, phase, y_true, y_pred):
    target_names = ['Excellent', 'Good', 'Average']
    print(f"\n{'='*30} {version_name} - Phase {phase} {'='*30}")
    print(classification_report(y_true, y_pred, digits=10, target_names=target_names))

    # G-Mean per class
    print("G-Mean per class (one-vs-rest):")
    for idx, name in enumerate(target_names):
        g = gmean_per_class(y_true, y_pred, idx)
        print(f"  {name:<10}: {g:.10f}")
        
    print()
    
    metrics = {
        'Version': version_name,
        'Phase': phase,
        'Accuracy': accuracy_score(y_true, y_pred),
        'BalancedAcc': balanced_accuracy_score(y_true, y_pred),
        'PrecMacro': precision_score(y_true, y_pred, average='macro'),
        'PrecWeighted': precision_score(y_true, y_pred, average='weighted'),
        'RecMacro': recall_score(y_true, y_pred, average='macro'),
        'RecWeighted': recall_score(y_true, y_pred, average='weighted'),
        'F1Macro': f1_score(y_true, y_pred, average='macro'),
        'F1Weighted': f1_score(y_true, y_pred, average='weighted'),
        'GMean': gmean_score(y_true, y_pred),
        'MCC': matthews_corrcoef(y_true, y_pred),
        'Kappa': cohen_kappa_score(y_true, y_pred)
    }
    
    for k, v in metrics.items():
        if k not in ['Version', 'Phase']:
            print(f"{k:18} : {v:.10f}")
    
    return metrics

# Chuẩn bị dữ liệu + train

In [4]:
def prepare_and_train_bilstm(train_path, val_path=None):
    df_train = pd.read_csv(train_path)

    # Nếu có validation thì gộp train + val để train chung
    if val_path:
        df_val = pd.read_csv(val_path)
        df = pd.concat([df_train, df_val], ignore_index=True)
    else:
        df = df_train

    df = df.drop(columns=['user_id', 'course_id'], errors='ignore')
    
    # Tách nhãn
    y = df['label_3'].values
    X_df = df.drop('label_3', axis=1)

    # Lấy các cột thuộc 4 phase (_1, _2, _3, _4)
    phase_cols = [c for c in X_df.columns if c.endswith(('_1', '_2', '_3', '_4'))]
     
    # Tên feature gốc (không bao gồm hậu tố phase)
    base_names = sorted(set(col.rsplit('_', 1)[0] for col in phase_cols))
    
    # Tạo dữ liệu chuỗi thời gian (4 phase)
    phases_data = []
    for p in ['1', '2', '3', '4']:
        cols_p = [c for c in phase_cols if c.endswith(f'_{p}')]
        phases_data.append(X_df[cols_p].values)
        
    # Shape: (samples, 4 phase, số feature mỗi phase)
    X = np.stack(phases_data, axis=1)
    print(f"X shape: {X.shape} → (samples, seq_len=4, features_per_phase)")

    # Chuẩn hóa dữ liệu
    scaler = StandardScaler()
    N, T, F = X.shape

    # Scale theo feature
    X_reshaped = X.reshape(-1, F)
    X_scaled = scaler.fit_transform(X_reshaped)
    X = X_scaled.reshape(N, T, F)
    
    le = LabelEncoder()
    y_enc = le.fit_transform(y)
    print(f"Classes: {le.classes_} → {le.transform(le.classes_)}")

    dataset = TimeSeriesBiLSTMDataset(X, y_enc)
    loader = DataLoader(dataset, batch_size=256, shuffle=True)

    model = TimeSeriesBiLSTM(
        input_dim_per_phase=F,
        hidden_dim=64,
        num_layers=2,
        dropout=0.3,
        bidirectional=True
    ).to(device)

    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=0.001)
    
    # Huấn luyện mô hình (có early stopping)
    model.train()
    best_loss = float('inf')
    patience = 10
    wait = 0

    for epoch in range(80):
        epoch_loss = 0

        for xb, yb in loader:
            xb, yb = xb.to(device), yb.to(device)

            optimizer.zero_grad()
            out = model(xb)
            loss = criterion(out, yb)
            loss.backward()
            optimizer.step()

            epoch_loss += loss.item()

        epoch_loss /= len(loader)
        
        # Lưu model tốt nhất
        if epoch_loss < best_loss - 1e-4:
            best_loss = epoch_loss
            best_state = model.state_dict()
            wait = 0
        else:
            wait += 1
            if wait >= patience:
                print(f"Early stopping at epoch {epoch+1}")
                break

    model.load_state_dict(best_state)
    model.eval()
    print("Training done")

    return model, scaler, le

# Chạy theo từng V

In [5]:
base_path = "/kaggle/input/mooccubex-data-cleaned/split_data"

In [6]:
def run_experiment(
    base_path,
    data_folder,
    train_file,
    val_file,
    test_prefix,
    version_name
):
    print(f"\n{'#'*5}")
    print(f"Version: {version_name}")
    print(f"{'#'*5}")

    train_path = f"{base_path}/{data_folder}/{train_file}"
    val_path   = f"{base_path}/{data_folder}/{val_file}"

    test_files = [
        f"{base_path}/{data_folder}/{test_prefix}_1.csv",
        f"{base_path}/{data_folder}/{test_prefix}_2.csv",
        f"{base_path}/{data_folder}/{test_prefix}_3.csv",
        f"{base_path}/{data_folder}/{test_prefix}_4.csv",
    ]

    # Huấn luyện mô hình BiLSTM
    model, scaler, le = prepare_and_train_bilstm(train_path, val_path)
    results = []
    model.eval()
    with torch.no_grad():

        # Đánh giá mô hình trên từng phase test
        for phase, test_path in enumerate(test_files, 1):
            print(f"\n--- Test Phase {phase}: {test_path} ---")

            df_test = pd.read_csv(test_path)

            df_test = df_test.drop(columns=['user_id', 'course_id'], errors='ignore')

            # Tách nhãn
            y_test = df_test['label_3'].values
            df_test = df_test.drop('label_3', axis=1)

            # Tách đặc trưng theo 4 phase
            phase_cols = [
                col for col in df_test.columns
                if col.endswith(('_1', '_2', '_3', '_4'))
            ]

            phases_data = []
            for p in ['1', '2', '3', '4']:
                cols_p = [c for c in phase_cols if c.endswith(f'_{p}')]
                phases_data.append(df_test[cols_p].values)

            # Dạng dữ liệu cuối: (số mẫu, seq_len=4, số đặc trưng mỗi phase)
            X_test = np.stack(phases_data, axis=1)

            # Chuẩn hóa dữ liệu (Dùng scaler đã fit từ tập train)
            N, T, F = X_test.shape
            X_test_scaled = scaler.transform(
                X_test.reshape(-1, F)
            ).reshape(N, T, F)

            # Dự đoán bằng mô hình BiLSTM
            X_tensor = torch.FloatTensor(X_test_scaled).to(device)
            preds = torch.argmax(model(X_tensor), dim=1).cpu().numpy()

            # Encode nhãn
            y_test_enc = le.transform(y_test)

            res = print_results(version_name, phase, y_test_enc, preds)
            results.append(res)

    return pd.DataFrame(results).round(10)

## V0 (Raw Median)

In [7]:
df_v0 = run_experiment(
    base_path=base_path,
    data_folder="data_median_im_3",
    train_file="train_us.csv",
    val_file="val.csv",
    test_prefix="test",
    version_name="V0 (Raw Median)"
)

df_v0


#####
Version: V0 (Raw Median)
#####
X shape: (439074, 4, 20) → (samples, seq_len=4, features_per_phase)
Classes: [0 1 2] → [0 1 2]
Training done

--- Test Phase 1: /kaggle/input/mooccubex-data-cleaned/split_data/data_median_im_3/test_1.csv ---

============================== V0 (Raw Median) - Phase 1 ==============================
              precision    recall  f1-score   support

   Excellent  0.0000000000 0.0000000000 0.0000000000       223
        Good  0.0026027103 1.0000000000 0.0051919075       605
     Average  1.0000000000 0.0000129520 0.0000259036    231625

    accuracy                      0.0026155825    232453
   macro avg  0.3342009034 0.3333376507 0.0017392704    232453
weighted avg  0.9964447636 0.0026155825 0.0000393242    232453

G-Mean per class (one-vs-rest):
  Excellent : 0.0000000000
  Good      : 0.0035971533
  Average   : 0.0035988845

Accuracy           : 0.0026155825
BalancedAcc        : 0.3333376507
PrecMacro          : 0.3342009034
PrecWeighted       :

,Version,Phase,Accuracy,BalancedAcc,PrecMacro,PrecWeighted,RecMacro,RecWeighted,F1Macro,F1Weighted,GMean,MCC,Kappa
0,V0 (Raw Median),1,0.002616,0.333338,0.334201,0.996445,0.333338,0.002616,0.001739,0.000039,0.000000,0.000186,7.980000e-08
1,V0 (Raw Median),2,0.003136,0.328566,0.298394,0.889445,0.328566,0.003136,0.002089,0.001157,0.000000,-0.033993,-1.039126e-04
2,V0 (Raw Median),3,0.994537,0.627006,0.643401,0.996761,0.627006,0.994537,0.570122,0.995330,0.675369,0.468618,4.490125e-01
3,V0 (Raw Median),4,0.997363,0.834401,0.742897,0.997895,0.834401,0.997363,0.782010,0.997579,0.881982,0.687299,6.805018e-01


## V1 (Median CDS)

In [8]:
df_v1 = run_experiment(
    base_path=base_path,
    data_folder="data_median_cdsmote",
    train_file="train_median_cdsmote.csv",
    val_file="val.csv",
    test_prefix="test",
    version_name="V1 (Median CDS)"
)

df_v1


#####
Version: V1 (Median CDS)
#####
X shape: (832452, 4, 20) → (samples, seq_len=4, features_per_phase)
Classes: [0 1 2] → [0 1 2]
Training done

--- Test Phase 1: /kaggle/input/mooccubex-data-cleaned/split_data/data_median_cdsmote/test_1.csv ---

============================== V1 (Median CDS) - Phase 1 ==============================
              precision    recall  f1-score   support

   Excellent  0.3225806452 0.1345291480 0.1898734177       223
        Good  0.0024746190 0.9504132231 0.0049363850       605
     Average  1.0000000000 0.0000043173 0.0000086346    231625

    accuracy                      0.0026069786    232453
   macro avg  0.4416850881 0.3616488961 0.0649394791    232453
weighted avg  0.9967538927 0.0026069786 0.0002036036    232453

G-Mean per class (one-vs-rest):
  Excellent : 0.3667323991
  Good      : 0.0161973716
  Average   : 0.0020778169

Accuracy           : 0.0026069786
BalancedAcc        : 0.3616488961
PrecMacro          : 0.4416850881
PrecWeighted     

,Version,Phase,Accuracy,BalancedAcc,PrecMacro,PrecWeighted,RecMacro,RecWeighted,F1Macro,F1Weighted,GMean,MCC,Kappa
0,V1 (Median CDS),1,0.002607,0.361649,0.441685,0.996754,0.361649,0.002607,0.064939,0.000204,0.023110,0.000285,6.858000e-07
1,V1 (Median CDS),2,0.003997,0.343238,0.413763,0.941531,0.343238,0.003997,0.028576,0.002872,0.067597,-0.015094,-7.269600e-05
2,V1 (Median CDS),3,0.514091,0.467502,0.421603,0.995387,0.467502,0.514091,0.276738,0.676234,0.505418,0.034121,4.167700e-03
3,V1 (Median CDS),4,0.997806,0.744474,0.808246,0.997643,0.744474,0.997806,0.773999,0.997710,0.790851,0.669149,6.670929e-01


## V2 (Median SAS)

In [9]:
df_v2 = run_experiment(
    base_path=base_path,
    data_folder="data_median_sasmote",
    train_file="train_median_sasmote.csv",
    val_file="val.csv",
    test_prefix="test",
    version_name="V2 (Median SAS)"
)

df_v2


#####
Version: V2 (Median SAS)
#####
X shape: (832452, 4, 20) → (samples, seq_len=4, features_per_phase)
Classes: [0 1 2] → [0 1 2]
Training done

--- Test Phase 1: /kaggle/input/mooccubex-data-cleaned/split_data/data_median_sasmote/test_1.csv ---

============================== V2 (Median SAS) - Phase 1 ==============================
              precision    recall  f1-score   support

   Excellent  0.3661971831 0.1165919283 0.1768707483       223
        Good  0.3891050584 0.1652892562 0.2320185615       605
     Average  0.9971007001 0.9992531031 0.9981757412    231625

    accuracy                      0.9962357982    232453
   macro avg  0.5841343138 0.4270447625 0.4690216837    232453
weighted avg  0.9949130370 0.9962357982 0.9953937763    232453

G-Mean per class (one-vs-rest):
  Excellent : 0.3414225181
  Good      : 0.4064201368
  Average   : 0.4325023121

Accuracy           : 0.9962357982
BalancedAcc        : 0.4270447625
PrecMacro          : 0.5841343138
PrecWeighted     

,Version,Phase,Accuracy,BalancedAcc,PrecMacro,PrecWeighted,RecMacro,RecWeighted,F1Macro,F1Weighted,GMean,MCC,Kappa
0,V2 (Median SAS),1,0.996236,0.427045,0.584134,0.994913,0.427045,0.996236,0.469022,0.995394,0.391518,0.268316,0.241830
1,V2 (Median SAS),2,0.993861,0.495360,0.480739,0.995215,0.495360,0.993861,0.479632,0.994484,0.542412,0.300124,0.294903
2,V2 (Median SAS),3,0.991641,0.682668,0.465191,0.996266,0.682668,0.991641,0.524113,0.993721,0.762579,0.382094,0.342430
3,V2 (Median SAS),4,0.995720,0.881063,0.629092,0.997641,0.881063,0.995720,0.716699,0.996456,0.921958,0.620888,0.588465


## V3 (Median Radius)

In [10]:
df_v3 = run_experiment(
    base_path=base_path,
    data_folder="data_median_radiussmote",
    train_file="train_median_radiussmote.csv",
    val_file="val.csv",
    test_prefix="test",
    version_name="V3 (Median Radius)"
)

df_v3


#####
Version: V3 (Median Radius)
#####
X shape: (832452, 4, 20) → (samples, seq_len=4, features_per_phase)
Classes: [0 1 2] → [0 1 2]
Training done

--- Test Phase 1: /kaggle/input/mooccubex-data-cleaned/split_data/data_median_radiussmote/test_1.csv ---

============================== V3 (Median Radius) - Phase 1 ==============================
              precision    recall  f1-score   support

   Excellent  0.0008822881 0.9147982063 0.0017628759       223
        Good  0.0000000000 0.0000000000 0.0000000000       605
     Average  0.9304207120 0.0049649217 0.0098771370    231625

    accuracy                      0.0058248334    232453
   macro avg  0.3104343333 0.3065877093 0.0038800043    232453
weighted avg  0.9271073901 0.0058248334 0.0098436457    232453

G-Mean per class (one-vs-rest):
  Excellent : 0.0692386801
  Good      : 0.0000000000
  Average   : 0.0667026347

Accuracy           : 0.0058248334
BalancedAcc        : 0.3065877093
PrecMacro          : 0.3104343333
PrecWei

,Version,Phase,Accuracy,BalancedAcc,PrecMacro,PrecWeighted,RecMacro,RecWeighted,F1Macro,F1Weighted,GMean,MCC,Kappa
0,V3 (Median Radius),1,0.005825,0.306588,0.310434,0.927107,0.306588,0.005825,0.003880,0.009844,0.000000,-0.049336,-0.000430
1,V3 (Median Radius),2,0.006896,0.309610,0.368453,0.898682,0.309610,0.006896,0.017506,0.011975,0.092982,-0.067173,-0.000668
2,V3 (Median Radius),3,0.009593,0.349982,0.426742,0.919218,0.349982,0.009593,0.087386,0.017068,0.148665,-0.026762,-0.000326
3,V3 (Median Radius),4,0.995186,0.847411,0.588937,0.997472,0.847411,0.995186,0.674388,0.996083,0.901854,0.589558,0.553705


## V4 (Tree CDS)

In [11]:
df_v4 = run_experiment(
    base_path=base_path,
    data_folder="data_extra_trees_cdsmote",
    train_file="train_extratrees_cdsmote.csv",
    val_file="val.csv",
    test_prefix="test",
    version_name="V4 (Tree CDS)"
)

df_v4


#####
Version: V4 (Tree CDS)
#####
X shape: (832452, 4, 20) → (samples, seq_len=4, features_per_phase)
Classes: [0 1 2] → [0 1 2]
Training done

--- Test Phase 1: /kaggle/input/mooccubex-data-cleaned/split_data/data_extra_trees_cdsmote/test_1.csv ---

============================== V4 (Tree CDS) - Phase 1 ==============================
              precision    recall  f1-score   support

   Excellent  0.0000000000 0.0000000000 0.0000000000       223
        Good  0.2000000000 0.0082644628 0.0158730159       605
     Average  0.9964935378 0.9999481921 0.9982178760    231625

    accuracy                      0.9964078760    232453
   macro avg  0.3988311793 0.3360708850 0.3380302973    232453
weighted avg  0.9934645528 0.9964078760 0.9947035259    232453

G-Mean per class (one-vs-rest):
  Excellent : 0.0000000000
  Good      : 0.0909051698
  Average   : 0.1252983227

Accuracy           : 0.9964078760
BalancedAcc        : 0.3360708850
PrecMacro          : 0.3988311793
PrecWeighted    

,Version,Phase,Accuracy,BalancedAcc,PrecMacro,PrecWeighted,RecMacro,RecWeighted,F1Macro,F1Weighted,GMean,MCC,Kappa
0,V4 (Tree CDS),1,0.996408,0.336071,0.398831,0.993465,0.336071,0.996408,0.338030,0.994704,0.000000,0.062111,0.020925
1,V4 (Tree CDS),2,0.996554,0.389424,0.474640,0.994700,0.389424,0.996554,0.413371,0.995480,0.000000,0.297900,0.248259
2,V4 (Tree CDS),3,0.996416,0.465547,0.580388,0.995746,0.465547,0.996416,0.484348,0.995969,0.446599,0.413769,0.407317
3,V4 (Tree CDS),4,0.994786,0.901394,0.574157,0.997596,0.901394,0.994786,0.669745,0.995858,0.936862,0.597059,0.550019


## V5 (Tree SAS)

In [12]:
df_v5 = run_experiment(
    base_path=base_path,
    data_folder="data_extra_trees_sasmote",
    train_file="train_extratrees_sasmote.csv",
    val_file="val.csv",
    test_prefix="test",
    version_name="V5 (Tree SAS)"
)

df_v5


#####
Version: V5 (Tree SAS)
#####
X shape: (832452, 4, 20) → (samples, seq_len=4, features_per_phase)
Classes: [0 1 2] → [0 1 2]
Training done

--- Test Phase 1: /kaggle/input/mooccubex-data-cleaned/split_data/data_extra_trees_sasmote/test_1.csv ---

============================== V5 (Tree SAS) - Phase 1 ==============================
              precision    recall  f1-score   support

   Excellent  0.0000000000 0.0000000000 0.0000000000       223
        Good  0.0000000000 0.0000000000 0.0000000000       605
     Average  0.9964379896 1.0000000000 0.9982158172    231625

    accuracy                      0.9964379896    232453
   macro avg  0.3321459965 0.3333333333 0.3327386057    232453
weighted avg  0.9928886671 0.9964379896 0.9946601621    232453

G-Mean per class (one-vs-rest):
  Excellent : 0.0000000000
  Good      : 0.0000000000
  Average   : 0.0000000000

Accuracy           : 0.9964379896
BalancedAcc        : 0.3333333333
PrecMacro          : 0.3321459965
PrecWeighted    

,Version,Phase,Accuracy,BalancedAcc,PrecMacro,PrecWeighted,RecMacro,RecWeighted,F1Macro,F1Weighted,GMean,MCC,Kappa
0,V5 (Tree SAS),1,0.996438,0.333333,0.332146,0.992889,0.333333,0.996438,0.332739,0.994660,0.000000,0.000000,0.000000
1,V5 (Tree SAS),2,0.995655,0.391321,0.409202,0.994220,0.391321,0.995655,0.399008,0.994927,0.000000,0.223471,0.214108
2,V5 (Tree SAS),3,0.993848,0.459003,0.457111,0.994754,0.459003,0.993848,0.454139,0.994278,0.484077,0.244841,0.242862
3,V5 (Tree SAS),4,0.995036,0.900701,0.587971,0.997579,0.900701,0.995036,0.684087,0.996000,0.935032,0.602215,0.559169


## V6 (Tree Radius)

In [13]:
df_v6 = run_experiment(
    base_path=base_path,
    data_folder="data_extra_trees_radiussmote",
    train_file="train_extratrees_radiussmote.csv",
    val_file="val.csv",
    test_prefix="test",
    version_name="V6 (Tree Radius)"
)

df_v6


#####
Version: V6 (Tree Radius)
#####
X shape: (832452, 4, 20) → (samples, seq_len=4, features_per_phase)
Classes: [0 1 2] → [0 1 2]
Training done

--- Test Phase 1: /kaggle/input/mooccubex-data-cleaned/split_data/data_extra_trees_radiussmote/test_1.csv ---

============================== V6 (Tree Radius) - Phase 1 ==============================
              precision    recall  f1-score   support

   Excellent  0.0000000000 0.0000000000 0.0000000000       223
        Good  0.0046558159 0.4000000000 0.0092044957       605
     Average  0.9973292700 0.7770879655 0.8735404028    231625

    accuracy                      0.7753610407    232453
   macro avg  0.3339950286 0.3923626552 0.2942482995    232453
weighted avg  0.9937888903 0.7753610407 0.8704527992    232453

G-Mean per class (one-vs-rest):
  Excellent : 0.0000000000
  Good      : 0.5574419434
  Average   : 0.5698466147

Accuracy           : 0.7753610407
BalancedAcc        : 0.3923626552
PrecMacro          : 0.3339950286
PrecWe

,Version,Phase,Accuracy,BalancedAcc,PrecMacro,PrecWeighted,RecMacro,RecWeighted,F1Macro,F1Weighted,GMean,MCC,Kappa
0,V6 (Tree Radius),1,0.775361,0.392363,0.333995,0.993789,0.392363,0.775361,0.294248,0.870453,0.000000,0.023178,0.005098
1,V6 (Tree Radius),2,0.778514,0.464856,0.335196,0.994837,0.464856,0.778514,0.296616,0.872503,0.000000,0.051091,0.011294
2,V6 (Tree Radius),3,0.985287,0.568336,0.539050,0.996293,0.568336,0.985287,0.404348,0.989977,0.407664,0.297389,0.230178
3,V6 (Tree Radius),4,0.994657,0.881400,0.570153,0.997516,0.881400,0.994657,0.663348,0.995760,0.925376,0.585448,0.539203
